# Experiment: Smart Warehouse Delay Prediction - Pre-Model Analysis

Objective:
- Understand the dataset structure before training any model.
- Build a reproducible preprocessing and feature-engineering workflow up to model-ready tables.
- Stop before model fitting, while keeping leakage risks and later validation needs explicit.


In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-smart-storage')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid')

SEED = 42
np.random.seed(SEED)

TARGET_COL = 'avg_delay_minutes_next_30m'
ID_COL = 'ID'
LAYOUT_KEY = 'layout_id'
GROUP_COL = 'scenario_id'


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() or ((candidate / 'data').exists() and (candidate / 'pyproject.toml').exists()):
            return candidate
    raise FileNotFoundError('Could not locate repo root from current working directory.')


def safe_divide(numerator, denominator):
    if isinstance(denominator, pd.Series):
        denominator = denominator.replace(0, np.nan)
    elif denominator == 0:
        denominator = np.nan
    return numerator / denominator


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / 'data'
REPO_ROOT


## Workflow

1. Load the raw tables and verify train/test alignment.
2. Join `layout_info` safely and inspect structural coverage.
3. Profile missing values, range issues, outliers, constants, and train-test shift.
4. Define preprocessing rules and create engineered features.
5. Produce `train_model_input` and `test_model_input` without fitting a model.


In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
layout_info = pd.read_csv(DATA_DIR / 'layout_info.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

raw_table_summary = pd.DataFrame(
    [
        {'table': 'train', 'rows': len(train), 'cols': train.shape[1]},
        {'table': 'test', 'rows': len(test), 'cols': test.shape[1]},
        {'table': 'layout_info', 'rows': len(layout_info), 'cols': layout_info.shape[1]},
        {'table': 'sample_submission', 'rows': len(sample_submission), 'cols': sample_submission.shape[1]},
    ]
)

raw_table_summary


In [ ]:
train_feature_cols = [col for col in train.columns if col != TARGET_COL]
test_feature_cols = list(test.columns)

missing_from_test = sorted(set(train_feature_cols) - set(test_feature_cols))
extra_in_test = sorted(set(test_feature_cols) - set(train_feature_cols))
assert not missing_from_test, f'Columns missing from test: {missing_from_test}'
assert not extra_in_test, f'Unexpected extra columns in test: {extra_in_test}'

layout_feature_cols = [col for col in layout_info.columns if col != LAYOUT_KEY]
meta_cols = [ID_COL, LAYOUT_KEY, GROUP_COL]
categorical_candidates = [LAYOUT_KEY, 'day_of_week', 'shift_hour']
numeric_feature_candidates = [
    col for col in train_feature_cols
    if col not in meta_cols and pd.api.types.is_numeric_dtype(train[col])
]

column_role_summary = pd.DataFrame(
    [
        {'role': 'id_only', 'columns': ID_COL},
        {'role': 'group_only', 'columns': GROUP_COL},
        {'role': 'layout_join_key', 'columns': LAYOUT_KEY},
        {'role': 'target', 'columns': TARGET_COL},
        {'role': 'categorical_candidates', 'columns': ', '.join(categorical_candidates)},
        {'role': 'layout_static_columns', 'columns': ', '.join(layout_feature_cols)},
        {'role': 'numeric_feature_count', 'columns': str(len(numeric_feature_candidates))},
    ]
)

key_cardinality = pd.Series(
    {
        'layout_id_unique_train': train[LAYOUT_KEY].nunique(dropna=True),
        'layout_id_unique_test': test[LAYOUT_KEY].nunique(dropna=True),
        'scenario_id_unique_train': train[GROUP_COL].nunique(dropna=True),
        'scenario_id_unique_test': test[GROUP_COL].nunique(dropna=True),
        'day_of_week_unique_train': train['day_of_week'].nunique(dropna=True),
        'shift_hour_unique_train': train['shift_hour'].nunique(dropna=True),
        'layout_type_unique': layout_info['layout_type'].nunique(dropna=True),
    },
    name='unique_count',
)

column_role_summary


In [ ]:
key_cardinality


## Analysis Helpers

These helpers generate diagnostics for missingness, outliers, range issues, constants, and train-test shift without fitting any model.


In [ ]:
def compute_psi(train_s: pd.Series, test_s: pd.Series, bins: int = 10) -> float:
    train_s = pd.to_numeric(train_s, errors='coerce').dropna()
    test_s = pd.to_numeric(test_s, errors='coerce').dropna()
    if train_s.empty or test_s.empty or train_s.nunique() < 5:
        return np.nan

    quantiles = np.unique(np.nanquantile(train_s, np.linspace(0, 1, bins + 1)))
    if len(quantiles) < 3:
        return np.nan

    bucket_edges = quantiles.astype(float).copy()
    bucket_edges[0] = -np.inf
    bucket_edges[-1] = np.inf

    train_bins = pd.cut(train_s, bucket_edges, include_lowest=True)
    test_bins = pd.cut(test_s, bucket_edges, include_lowest=True)

    aligned = pd.concat(
        [
            train_bins.value_counts(normalize=True, sort=False),
            test_bins.value_counts(normalize=True, sort=False),
        ],
        axis=1,
    ).fillna(0.0)
    aligned.columns = ['train_pct', 'test_pct']

    epsilon = 1e-6
    train_pct = aligned['train_pct'].clip(lower=epsilon)
    test_pct = aligned['test_pct'].clip(lower=epsilon)
    return float(((train_pct - test_pct) * np.log(train_pct / test_pct)).sum())


def summarize_missing(train_df: pd.DataFrame, test_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        rows.append(
            {
                'column': col,
                'train_missing_rate': train_df[col].isna().mean() if col in train_df.columns else np.nan,
                'test_missing_rate': test_df[col].isna().mean() if col in test_df.columns else np.nan,
                'missing_rate_gap': (
                    abs(train_df[col].isna().mean() - test_df[col].isna().mean())
                    if col in train_df.columns and col in test_df.columns
                    else np.nan
                ),
            }
        )
    return pd.DataFrame(rows).sort_values(['train_missing_rate', 'missing_rate_gap'], ascending=False)


def build_constant_report(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        series = df[col]
        value_share = series.value_counts(dropna=False, normalize=True)
        top_share = float(value_share.iloc[0]) if not value_share.empty else np.nan
        nunique = int(series.nunique(dropna=True))
        rows.append(
            {
                'column': col,
                'nunique_non_null': nunique,
                'top_value_share': top_share,
                'is_constant': nunique <= 1,
                'is_near_constant': top_share >= 0.995 if pd.notna(top_share) else False,
            }
        )
    return pd.DataFrame(rows).sort_values(['is_constant', 'is_near_constant', 'top_value_share'], ascending=False)


def propose_bounds(column: str):
    bounded_zero_one = {
        'daily_forecast_accuracy',
        'pack_utilization',
        'loading_dock_util',
        'express_lane_util',
        'staging_area_util',
        'vertical_utilization',
        'agv_task_success_rate',
        'barcode_read_success_rate',
    }
    if column.endswith('_pct'):
        return (0.0, 100.0)
    if column.endswith('_ratio') or column.endswith('_util') or column.endswith('_utilization') or column.endswith('_success_rate'):
        return (0.0, 1.0)
    if column in bounded_zero_one:
        return (0.0, 1.0)
    return None


def build_range_violation_report(train_df: pd.DataFrame, test_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        bounds = propose_bounds(col)
        if bounds is None:
            continue
        lower, upper = bounds
        train_s = pd.to_numeric(train_df[col], errors='coerce')
        test_s = pd.to_numeric(test_df[col], errors='coerce')
        train_mask = train_s.notna()
        test_mask = test_s.notna()
        train_rate = ((train_s[train_mask] < lower) | (train_s[train_mask] > upper)).mean() if train_mask.any() else np.nan
        test_rate = ((test_s[test_mask] < lower) | (test_s[test_mask] > upper)).mean() if test_mask.any() else np.nan
        rows.append(
            {
                'column': col,
                'expected_bounds': f'[{lower}, {upper}]',
                'train_out_of_bounds_rate': train_rate,
                'test_out_of_bounds_rate': test_rate,
            }
        )
    if not rows:
        return pd.DataFrame(columns=['column', 'expected_bounds', 'train_out_of_bounds_rate', 'test_out_of_bounds_rate'])
    return pd.DataFrame(rows).sort_values(['train_out_of_bounds_rate', 'test_out_of_bounds_rate'], ascending=False)


def build_outlier_report(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if series.empty:
            continue

        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr > 0:
            lower_iqr = q1 - 1.5 * iqr
            upper_iqr = q3 + 1.5 * iqr
            pct_outside_iqr = ((series < lower_iqr) | (series > upper_iqr)).mean()
        else:
            lower_iqr = np.nan
            upper_iqr = np.nan
            pct_outside_iqr = 0.0

        median = series.median()
        mad = (series - median).abs().median()
        if mad > 0:
            robust_z = 0.6745 * (series - median) / mad
            pct_robust_gt4 = (robust_z.abs() > 4).mean()
        else:
            pct_robust_gt4 = np.nan

        rows.append(
            {
                'column': col,
                'p01': series.quantile(0.01),
                'p50': median,
                'p99': series.quantile(0.99),
                'pct_outside_iqr': pct_outside_iqr,
                'pct_robust_z_gt4': pct_robust_gt4,
                'lower_iqr_bound': lower_iqr,
                'upper_iqr_bound': upper_iqr,
                'clipping_candidate': bool((pct_outside_iqr > 0.01) and (series.nunique() > 20)),
            }
        )
    if not rows:
        return pd.DataFrame(columns=['column', 'p01', 'p50', 'p99', 'pct_outside_iqr', 'pct_robust_z_gt4', 'lower_iqr_bound', 'upper_iqr_bound', 'clipping_candidate'])
    return pd.DataFrame(rows).sort_values(['pct_outside_iqr', 'pct_robust_z_gt4'], ascending=False)


def build_train_test_shift_report(train_df: pd.DataFrame, test_df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in numeric_cols:
        train_s = pd.to_numeric(train_df[col], errors='coerce')
        test_s = pd.to_numeric(test_df[col], errors='coerce')
        train_non_null = train_s.dropna()
        test_non_null = test_s.dropna()
        if train_non_null.empty or test_non_null.empty:
            continue

        pooled_std = np.sqrt((train_non_null.std() ** 2 + test_non_null.std() ** 2) / 2)
        standardized_mean_gap = abs(train_non_null.mean() - test_non_null.mean()) / pooled_std if pooled_std and pooled_std > 0 else np.nan
        rows.append(
            {
                'column': col,
                'train_mean': train_non_null.mean(),
                'test_mean': test_non_null.mean(),
                'train_median': train_non_null.median(),
                'test_median': test_non_null.median(),
                'standardized_mean_gap': standardized_mean_gap,
                'psi': compute_psi(train_non_null, test_non_null),
            }
        )
    if not rows:
        return pd.DataFrame(columns=['column', 'train_mean', 'test_mean', 'train_median', 'test_median', 'standardized_mean_gap', 'psi'])
    return pd.DataFrame(rows).sort_values(['psi', 'standardized_mean_gap'], ascending=False)


def propose_missing_strategy(column: str, missing_rate: float) -> str:
    if pd.isna(missing_rate) or missing_rate == 0:
        return 'keep_as_is'

    column_lower = column.lower()
    if column == 'layout_type':
        return 'missing_indicator_candidate'
    if any(token in column_lower for token in ['ratio', 'util', 'pct', 'success_rate', 'accuracy']):
        return 'missing_indicator_candidate'
    if any(token in column_lower for token in ['fault', 'collision', 'blocked', 'override', 'backorder']) and missing_rate < 0.30:
        return 'zero_impute_candidate'
    if any(token in column_lower for token in ['temp', 'speed', 'time', 'distance', 'weight', 'age', 'power', 'level', 'hours', 'minute', 'wait', 'volume', 'length', 'count', 'density', 'queue', 'score']):
        return 'median_impute_candidate'
    return 'review_manually'


## Join `layout_info` and Diagnose the Raw Feature Space


In [ ]:
layout_duplicates = layout_info[layout_info.duplicated(LAYOUT_KEY, keep=False)]
assert layout_duplicates.empty, 'layout_info contains duplicated layout_id values.'

layout_coverage_summary = pd.DataFrame(
    [
        {
            'dataset': 'train',
            'coverage_rate': train[LAYOUT_KEY].isin(layout_info[LAYOUT_KEY]).mean(),
            'missing_layouts': train.loc[~train[LAYOUT_KEY].isin(layout_info[LAYOUT_KEY]), LAYOUT_KEY].nunique(),
        },
        {
            'dataset': 'test',
            'coverage_rate': test[LAYOUT_KEY].isin(layout_info[LAYOUT_KEY]).mean(),
            'missing_layouts': test.loc[~test[LAYOUT_KEY].isin(layout_info[LAYOUT_KEY]), LAYOUT_KEY].nunique(),
        },
    ]
)

train_merged = train.merge(layout_info, on=LAYOUT_KEY, how='left', validate='many_to_one')
test_merged = test.merge(layout_info, on=LAYOUT_KEY, how='left', validate='many_to_one')

assert len(train_merged) == len(train)
assert len(test_merged) == len(test)

merged_feature_cols = [col for col in train_merged.columns if col != TARGET_COL]
merged_numeric_cols = [
    col for col in merged_feature_cols
    if pd.api.types.is_numeric_dtype(train_merged[col]) and col != ID_COL
]

layout_coverage_summary


In [ ]:
missing_report = summarize_missing(train_merged.drop(columns=[TARGET_COL]), test_merged, merged_feature_cols)
constant_report = build_constant_report(train_merged.drop(columns=[TARGET_COL]), merged_feature_cols)
range_violation_report = build_range_violation_report(train_merged.drop(columns=[TARGET_COL]), test_merged, merged_numeric_cols)
outlier_report = build_outlier_report(train_merged, merged_numeric_cols)
shift_report = build_train_test_shift_report(train_merged.drop(columns=[TARGET_COL]), test_merged, merged_numeric_cols)

missing_report.head(15)


In [ ]:
diagnostic_snapshot = {
    'columns_with_missing_values': int((missing_report['train_missing_rate'] > 0).sum()),
    'constant_columns': int(constant_report['is_constant'].sum()),
    'near_constant_columns': int(constant_report['is_near_constant'].sum()),
    'bounded_columns_checked': int(len(range_violation_report)),
    'columns_flagged_for_clipping_review': int(outlier_report['clipping_candidate'].sum()),
}

diagnostic_snapshot


## Lightweight EDA

The goal here is to understand target behavior, missingness concentration, and a few high-signal raw relationships before any preprocessing decisions are locked in.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_merged[TARGET_COL], bins=50, ax=axes[0], color='#3b7ddd')
axes[0].set_title('Target Distribution')
axes[0].set_xlabel(TARGET_COL)

top_missing = missing_report.head(15).sort_values('train_missing_rate')
axes[1].barh(top_missing['column'], top_missing['train_missing_rate'], color='#f28e2b')
axes[1].set_title('Top Missing Rates in Train')
axes[1].set_xlabel('Missing Rate')

plt.tight_layout()
plt.show()

target_summary = train_merged[TARGET_COL].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame('target_stat')
target_summary


In [ ]:
target_correlations = (
    train_merged[[TARGET_COL] + merged_numeric_cols]
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL)
    .rename('corr_with_target')
    .to_frame()
)
target_correlations['abs_corr'] = target_correlations['corr_with_target'].abs()
target_correlations = target_correlations.sort_values('abs_corr', ascending=False)

group_target_summary = {
    'day_of_week': train_merged.groupby('day_of_week', dropna=False)[TARGET_COL].agg(['count', 'mean', 'median']).sort_index(),
    'shift_hour': train_merged.groupby('shift_hour', dropna=False)[TARGET_COL].agg(['count', 'mean', 'median']).sort_index(),
    'layout_type': train_merged.groupby('layout_type', dropna=False)[TARGET_COL].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False),
}

target_correlations.head(20)


In [ ]:
group_target_summary['layout_type']


## Preprocessing Policy

This stage defines what should be kept, deferred, or flagged. It does not yet perform model-specific encoding or imputation.


In [ ]:
preprocessing_plan = pd.DataFrame({'column': merged_feature_cols})
preprocessing_plan = preprocessing_plan.merge(missing_report, on='column', how='left')
preprocessing_plan = preprocessing_plan.merge(
    constant_report[['column', 'nunique_non_null', 'top_value_share', 'is_constant', 'is_near_constant']],
    on='column',
    how='left',
)
preprocessing_plan = preprocessing_plan.merge(
    range_violation_report[['column', 'expected_bounds', 'train_out_of_bounds_rate', 'test_out_of_bounds_rate']],
    on='column',
    how='left',
)

def infer_modeling_role(column: str) -> str:
    if column == ID_COL:
        return 'id_only'
    if column == GROUP_COL:
        return 'group_only'
    if column in [LAYOUT_KEY, 'layout_type', 'day_of_week', 'shift_hour']:
        return 'categorical_candidate'
    if column in layout_feature_cols:
        return 'layout_static_numeric'
    return 'numeric_feature'


preprocessing_plan['modeling_role'] = preprocessing_plan['column'].map(infer_modeling_role)
preprocessing_plan['proposed_missing_strategy'] = preprocessing_plan.apply(
    lambda row: propose_missing_strategy(row['column'], row['train_missing_rate']),
    axis=1,
)
preprocessing_plan['use_for_modeling_now'] = ~preprocessing_plan['column'].isin([ID_COL, GROUP_COL])
preprocessing_plan['needs_manual_review'] = (
    preprocessing_plan['is_constant'].fillna(False)
    | preprocessing_plan['is_near_constant'].fillna(False)
    | (preprocessing_plan['train_out_of_bounds_rate'].fillna(0) > 0)
)

preprocessing_plan.sort_values(['needs_manual_review', 'train_missing_rate'], ascending=[False, False]).head(25)


## Feature Engineering

The engineered features below are deliberately leakage-safe: they only use concurrent row information and static layout metadata. No target statistics or scenario-level aggregates are used.


In [ ]:
ENGINEERED_FEATURE_SPECS = [
    {'feature_name': 'observed_robot_pool', 'feature_group': 'robot_state', 'source_columns': 'robot_active, robot_idle, robot_charging', 'description': 'Observed robots active in the snapshot.'},
    {'feature_name': 'robot_active_share_observed', 'feature_group': 'robot_state', 'source_columns': 'robot_active, robot_idle, robot_charging', 'description': 'Share of active robots among observed robots.'},
    {'feature_name': 'robot_idle_share_observed', 'feature_group': 'robot_state', 'source_columns': 'robot_active, robot_idle, robot_charging', 'description': 'Share of idle robots among observed robots.'},
    {'feature_name': 'robot_charging_share_observed', 'feature_group': 'robot_state', 'source_columns': 'robot_active, robot_idle, robot_charging', 'description': 'Share of charging robots among observed robots.'},
    {'feature_name': 'order_inflow_per_observed_robot', 'feature_group': 'capacity_pressure', 'source_columns': 'order_inflow_15m, observed_robot_pool', 'description': 'Recent order inflow normalized by observed robots.'},
    {'feature_name': 'order_inflow_per_layout_robot', 'feature_group': 'capacity_pressure', 'source_columns': 'order_inflow_15m, robot_total', 'description': 'Recent order inflow normalized by layout robot capacity.'},
    {'feature_name': 'unique_sku_per_layout_robot', 'feature_group': 'capacity_pressure', 'source_columns': 'unique_sku_15m, robot_total', 'description': 'SKU complexity per layout robot.'},
    {'feature_name': 'order_inflow_per_pack_station', 'feature_group': 'capacity_pressure', 'source_columns': 'order_inflow_15m, pack_station_count', 'description': 'Recent order inflow normalized by packing stations.'},
    {'feature_name': 'order_inflow_per_charger', 'feature_group': 'capacity_pressure', 'source_columns': 'order_inflow_15m, charger_count', 'description': 'Recent order inflow normalized by charger availability.'},
    {'feature_name': 'staff_per_1000sqm', 'feature_group': 'layout_normalized', 'source_columns': 'staff_on_floor, floor_area_sqm', 'description': 'Workforce density per 1,000 square meters.'},
    {'feature_name': 'robot_density_per_1000sqm', 'feature_group': 'layout_normalized', 'source_columns': 'robot_total, floor_area_sqm', 'description': 'Robot density per 1,000 square meters.'},
    {'feature_name': 'pack_station_density_per_1000sqm', 'feature_group': 'layout_normalized', 'source_columns': 'pack_station_count, floor_area_sqm', 'description': 'Packing station density per 1,000 square meters.'},
    {'feature_name': 'charge_queue_per_charger', 'feature_group': 'battery_stress', 'source_columns': 'charge_queue_length, charger_count', 'description': 'Charging queue normalized by charger count.'},
    {'feature_name': 'low_battery_charge_pressure', 'feature_group': 'battery_stress', 'source_columns': 'low_battery_ratio, charge_queue_length', 'description': 'Joint battery stress indicator.'},
    {'feature_name': 'battery_cv', 'feature_group': 'battery_stress', 'source_columns': 'battery_std, battery_mean', 'description': 'Battery coefficient of variation.'},
    {'feature_name': 'congestion_x_order_inflow', 'feature_group': 'congestion_interaction', 'source_columns': 'congestion_score, order_inflow_15m', 'description': 'Interaction between operational load and congestion.'},
    {'feature_name': 'dock_load_pressure', 'feature_group': 'throughput_interaction', 'source_columns': 'loading_dock_util, order_inflow_15m', 'description': 'Dock usage weighted by recent order inflow.'},
    {'feature_name': 'pack_load_pressure', 'feature_group': 'throughput_interaction', 'source_columns': 'pack_utilization, order_inflow_15m', 'description': 'Packing utilization weighted by recent order inflow.'},
    {'feature_name': 'blocked_paths_per_intersection', 'feature_group': 'congestion_interaction', 'source_columns': 'blocked_path_15m, intersection_count', 'description': 'Blocked paths normalized by layout intersections.'},
    {'feature_name': 'intersection_wait_x_aisle_traffic', 'feature_group': 'congestion_interaction', 'source_columns': 'intersection_wait_time_avg, aisle_traffic_score', 'description': 'Wait pressure under aisle traffic.'},
    {'feature_name': 'prev_shift_volume_per_1000sqm', 'feature_group': 'layout_normalized', 'source_columns': 'prev_shift_volume, floor_area_sqm', 'description': 'Previous shift volume per 1,000 square meters.'},
    {'feature_name': 'storage_x_vertical_pressure', 'feature_group': 'layout_normalized', 'source_columns': 'storage_density_pct, vertical_utilization', 'description': 'Combined static storage pressure indicator.'},
    {'feature_name': 'cold_chain_order_pressure', 'feature_group': 'capacity_pressure', 'source_columns': 'cold_chain_ratio, order_inflow_15m', 'description': 'Cold-chain order pressure.'},
    {'feature_name': 'shift_hour_sin', 'feature_group': 'temporal', 'source_columns': 'shift_hour', 'description': 'Cyclical shift encoding using sine.'},
    {'feature_name': 'shift_hour_cos', 'feature_group': 'temporal', 'source_columns': 'shift_hour', 'description': 'Cyclical shift encoding using cosine.'},
    {'feature_name': 'day_shift_key', 'feature_group': 'temporal', 'source_columns': 'day_of_week, shift_hour', 'description': 'Categorical interaction between day-of-week and shift-hour.'}
]


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    observed_robot_pool = df['robot_active'] + df['robot_idle'] + df['robot_charging']
    df['observed_robot_pool'] = observed_robot_pool
    df['robot_active_share_observed'] = safe_divide(df['robot_active'], observed_robot_pool)
    df['robot_idle_share_observed'] = safe_divide(df['robot_idle'], observed_robot_pool)
    df['robot_charging_share_observed'] = safe_divide(df['robot_charging'], observed_robot_pool)

    df['order_inflow_per_observed_robot'] = safe_divide(df['order_inflow_15m'], df['observed_robot_pool'])
    df['order_inflow_per_layout_robot'] = safe_divide(df['order_inflow_15m'], df['robot_total'])
    df['unique_sku_per_layout_robot'] = safe_divide(df['unique_sku_15m'], df['robot_total'])
    df['order_inflow_per_pack_station'] = safe_divide(df['order_inflow_15m'], df['pack_station_count'])
    df['order_inflow_per_charger'] = safe_divide(df['order_inflow_15m'], df['charger_count'])

    df['staff_per_1000sqm'] = safe_divide(df['staff_on_floor'] * 1000, df['floor_area_sqm'])
    df['robot_density_per_1000sqm'] = safe_divide(df['robot_total'] * 1000, df['floor_area_sqm'])
    df['pack_station_density_per_1000sqm'] = safe_divide(df['pack_station_count'] * 1000, df['floor_area_sqm'])

    df['charge_queue_per_charger'] = safe_divide(df['charge_queue_length'], df['charger_count'])
    df['low_battery_charge_pressure'] = df['low_battery_ratio'] * df['charge_queue_length']
    df['battery_cv'] = safe_divide(df['battery_std'], df['battery_mean'])

    df['congestion_x_order_inflow'] = df['congestion_score'] * df['order_inflow_15m']
    df['dock_load_pressure'] = df['loading_dock_util'] * df['order_inflow_15m']
    df['pack_load_pressure'] = df['pack_utilization'] * df['order_inflow_15m']
    df['blocked_paths_per_intersection'] = safe_divide(df['blocked_path_15m'], df['intersection_count'])
    df['intersection_wait_x_aisle_traffic'] = df['intersection_wait_time_avg'] * df['aisle_traffic_score']

    df['prev_shift_volume_per_1000sqm'] = safe_divide(df['prev_shift_volume'] * 1000, df['floor_area_sqm'])
    df['storage_x_vertical_pressure'] = df['storage_density_pct'] * df['vertical_utilization']
    df['cold_chain_order_pressure'] = df['cold_chain_ratio'] * df['order_inflow_15m']

    shift_hour = pd.to_numeric(df['shift_hour'], errors='coerce')
    df['shift_hour_sin'] = np.sin(2 * np.pi * shift_hour / 24)
    df['shift_hour_cos'] = np.cos(2 * np.pi * shift_hour / 24)
    day_text = pd.to_numeric(df['day_of_week'], errors='coerce').astype('Int64').astype(str)
    shift_text = shift_hour.astype('Int64').astype(str)
    df['day_shift_key'] = 'dow_' + day_text + '__shift_' + shift_text
    return df


train_features = add_engineered_features(train_merged)
test_features = add_engineered_features(test_merged)

engineered_feature_names = [spec['feature_name'] for spec in ENGINEERED_FEATURE_SPECS]
engineered_missing_report = summarize_missing(train_features.drop(columns=[TARGET_COL]), test_features, engineered_feature_names)
feature_summary = pd.DataFrame(ENGINEERED_FEATURE_SPECS).merge(
    engineered_missing_report[['column', 'train_missing_rate', 'test_missing_rate']],
    left_on='feature_name',
    right_on='column',
    how='left',
).drop(columns=['column'])

feature_summary


In [ ]:
excluded_from_modeling = [ID_COL, GROUP_COL, TARGET_COL]
feature_columns = [col for col in train_features.columns if col not in excluded_from_modeling]

train_target = train_features[TARGET_COL].copy()
train_groups = train_features[GROUP_COL].copy()
train_model_input = train_features[feature_columns].copy()
test_model_input = test_features[feature_columns].copy()

assert list(train_model_input.columns) == list(test_model_input.columns)
assert ID_COL not in train_model_input.columns
assert GROUP_COL not in train_model_input.columns
assert TARGET_COL not in train_model_input.columns
assert TARGET_COL not in test_model_input.columns

categorical_feature_candidates = [
    col for col in [LAYOUT_KEY, 'layout_type', 'day_of_week', 'shift_hour', 'day_shift_key']
    if col in train_model_input.columns
]

final_snapshot = {
    'train_model_input_shape': train_model_input.shape,
    'test_model_input_shape': test_model_input.shape,
    'target_shape': train_target.shape,
    'group_count': int(train_groups.nunique(dropna=True)),
    'engineered_feature_count': len(engineered_feature_names),
    'categorical_feature_candidates': categorical_feature_candidates,
}

final_snapshot


In [ ]:
train_model_input.head(3)


## Leakage Review and Next Actions

- `scenario_id` is kept only as a grouping variable for later validation and is excluded from the feature matrix.
- No target encoding, target aggregation, or scenario-level aggregate features are used.
- Missing-value handling is documented as a policy table first; model-specific imputation and encoding should be decided during the modeling stage.
- The next notebook iteration can start from `train_model_input`, `test_model_input`, `train_target`, and `train_groups` to compare grouped validation schemes and baseline models.
